**install scispacy model and import necessary libraries**

In [ ]:
#!pip install spacy

In [ ]:
pip install spacy==3.4.4

  Using cached spacy-3.4.4.tar.gz (1.2 MB)
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
!pip install scispacy

In [ ]:
!pip install scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_ner_bc5cdr_md-0.5.1.tar.gz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.2/120.2 MB 7.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Using cached spacy-3.4.4.tar.gz (1.2 MB)
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Installing build dependencies ... error
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [ ]:
import spacy
import en_ner_bionlp13cg_md

ModuleNotFoundError: No module named 'en_ner_bionlp13cg_md'

In [ ]:
! pip install en_ner_bionlp13cg_md

In [ ]:
nlp = en_ner_bionlp13cg_md.load()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.manifold import TSNE

from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize
from nltk.stem import WordNetLemmatizer

from imblearn.over_sampling import SMOTE

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tboyle10/medicaltranscriptions")

print("Path to dataset files:", path)

In [ ]:
import os

In [ ]:
print(os.listdir(path))

**Load dataset**

In [ ]:
mtdata_df = pd.read_csv(os.path.join(path, "mtsamples.csv"), index_col=0)

print(mtdata_df.columns)
mtdata_df.head(5)

In [ ]:
mtdata_df.info()

In [ ]:
mtdata_df.isna().sum()

In [ ]:
mtdata_df= mtdata_df.dropna(axis = 0, how ='any')

In [ ]:
mtdata_df.isna().sum()

In [ ]:
mtdata_df['medical_specialty'].unique()

In [ ]:
# List of columns to remove
columns_to_remove = ['description', 'sample_name' , 'keywords']

# Remove the specified columns
mtdata_df = mtdata_df.drop(columns=columns_to_remove, axis=1)

In [ ]:
mtdata_df.head()

In [ ]:
medical_specialty_counts = mtdata_df['medical_specialty'].value_counts()

# Create a bar chart
plt.figure(figsize=(10, 6))
medical_specialty_counts.plot(kind='bar')
plt.title('Medical Specialty Count')
plt.xlabel('Medical Specialty')
plt.ylabel('Count')
plt.xticks(rotation=90)  # Rotate x-axis labels for better visibility

plt.tight_layout()
plt.show()

In [ ]:
mtdata_df['transcription'][0]

In [ ]:
mtdata_df['transcription']=mtdata_df['transcription'].astype('str')
mtdata_df['transcription']

In [ ]:
mtdata_df['transcription'] = mtdata_df['transcription'].str.lower()
mtdata_df['transcription']

**Cleaning**

In [ ]:
def clean_text(text ):
    text = text.translate(str.maketrans('', '', string.punctuation))
    text1 = ''.join([w for w in text if not w.isdigit()])
    REPLACE_BY_SPACE_RE = re.compile('[/(){}\[\]\|@,;]')


    text2 = text1.lower()
    text2 = REPLACE_BY_SPACE_RE.sub('', text2)
    return text2

In [ ]:
mtdata_df['transcription'] = mtdata_df['transcription'].apply(clean_text)

In [ ]:
mtdata_df['transcription'][0]

In [ ]:
mtdata_df.head(5)

In [ ]:
mtdata_df['transcription'][0]

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('wordnet')

**Lemmatization**

In [ ]:
# Initialize the lemmatizer
nltk.download('wordnet')  # Download WordNet data
lemmatizer = WordNetLemmatizer()

# Function to lemmatize a sentence
def lemmatize_sentence(sentence):
    words = nltk.word_tokenize(sentence)
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(lemmatized_words)

# Lemmatize the 'transcription' column
mtdata_df['transcription'] = mtdata_df['transcription'].apply(lemmatize_sentence)

In [ ]:
mtdata_df['transcription'][0]

**Apply Scispacy model**

In [ ]:
def process_Text( text):
    wordlist=[]
    doc = nlp(text)
    for ent in doc.ents:
        wordlist.append(ent.text)
    return ' '.join(wordlist)

In [ ]:
mtdata_df['transcription'] = mtdata_df['transcription'].apply(process_Text)

In [ ]:
mtdata_df['transcription'][0]

**create a copy of the mtdata_df as mt_dt**

In [ ]:
mt_df = mtdata_df.copy()

In [ ]:
mt_df.info()

**Table showing average cosine similarity between different medical specialties based on the words present in their respective transcriptions**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tabulate import tabulate

# Create a dictionary to store word frequencies for each specialty
specialty_word_frequencies = {}

# Create a CountVectorizer to convert text data into word frequency vectors
vectorizer = CountVectorizer()

# Calculate word frequencies for each specialty
for specialty in mt_df['medical_specialty'].unique():
    specialty_texts = mt_df[mt_df['medical_specialty'] == specialty]['transcription']
    word_matrix = vectorizer.fit_transform(specialty_texts)
    word_frequencies = word_matrix.sum(axis=0)
    words = vectorizer.get_feature_names_out()
    specialty_word_frequencies[specialty] = dict(zip(words, word_frequencies.tolist()[0]))

# Create a list of all unique words across all specialties
all_unique_words = set(word for freq_dict in specialty_word_frequencies.values() for word in freq_dict.keys())

# Create word frequency array for each specialty
frequencies = []
for specialty, freq_dict in specialty_word_frequencies.items():
    freq_vector = [freq_dict[word] if word in freq_dict else 0 for word in all_unique_words]
    frequencies.append(freq_vector)

# Convert the frequencies list into a numpy array
frequencies_array = np.array(frequencies)

# Calculate cosine similarity between specialties
cosine_similarities = cosine_similarity(frequencies_array)

# Get the specialty names
specialties = list(specialty_word_frequencies.keys())

# Calculate the average cosine similarity for each specialty
avg_similarities = np.mean(cosine_similarities, axis=0)

# Create a list to store specialty information
specialty_info = []

# Populate the specialty information list
for specialty, avg_similarity in zip(specialties, avg_similarities):
    num_rows = len(mt_df[mt_df['medical_specialty'] == specialty])
    specialty_info.append([specialty, avg_similarity, num_rows])

# Sort the specialties by average cosine similarity in descending order
sorted_specialties = sorted(specialty_info, key=lambda x: x[1], reverse=True)

# Print the specialties table
headers = ["Medical Specialty", "Average Similarity", "Number of Rows"]
print(tabulate(sorted_specialties, headers=headers, tablefmt="pretty"))

**Remove some medical specialties with high average similarity**

In [ ]:
mt_df = mt_df[mt_df['medical_specialty'] != ' Consult - History and Phy.']
mt_df = mt_df[mt_df['medical_specialty'] != ' Surgery']
mt_df = mt_df[mt_df['medical_specialty'] != ' Emergency Room Reports']
mt_df = mt_df[mt_df['medical_specialty'] != ' SOAP / Chart / Progress Notes']
mt_df = mt_df[mt_df['medical_specialty'] != ' General Medicine']
mt_df = mt_df[mt_df['medical_specialty'] != ' Discharge Summary']
mt_df = mt_df[mt_df['medical_specialty'] != ' Hematology - Oncology']
mt_df = mt_df[mt_df['medical_specialty'] != ' Pediatrics - Neonatal']
mt_df = mt_df[mt_df['medical_specialty'] != ' Pain Management']
mt_df = mt_df[mt_df['medical_specialty'] != ' Psychiatry / Psychology']
mt_df = mt_df[mt_df['medical_specialty'] != ' IME-QME-Work Comp etc.']
mt_df = mt_df[mt_df['medical_specialty'] != ' Speech - Language']
mt_df = mt_df[mt_df['medical_specialty'] != ' Orthopedic']
mt_df = mt_df[mt_df['medical_specialty'] != ' Hospice - Palliative Care']
mt_df = mt_df[mt_df['medical_specialty'] != ' Sleep Medicine']
mt_df = mt_df[mt_df['medical_specialty'] != ' Letters']
mt_df = mt_df[mt_df['medical_specialty'] != ' Office Notes']
mt_df = mt_df[mt_df['medical_specialty'] != ' Chiropractic']
mt_df = mt_df[mt_df['medical_specialty'] != ' Bariatrics']
mt_df = mt_df[mt_df['medical_specialty'] != ' Diets and Nutritions']
mt_df = mt_df[mt_df['medical_specialty'] != ' Dentistry']
mt_df = mt_df[mt_df['medical_specialty'] != ' Cosmetic / Plastic Surgery']
mt_df = mt_df[mt_df['medical_specialty'] != ' Endocrinology']
mt_df = mt_df[mt_df['medical_specialty'] != ' Physical Medicine - Rehab']
mt_df = mt_df[mt_df['medical_specialty'] != ' Rheumatology']
mt_df = mt_df[mt_df['medical_specialty'] != ' Allergy / Immunology']
mt_df = mt_df[mt_df['medical_specialty'] != ' Lab Medicine - Pathology']

**Heat map showing count of common words between each medical specialties**

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Create a dictionary to store word frequencies for each specialty
specialty_word_frequencies = {}

# Create a CountVectorizer to convert text data into word frequency vectors
vectorizer = CountVectorizer()

# Calculate word frequencies for each specialty
for specialty in mt_df['medical_specialty'].unique():
    specialty_texts = mt_df[mt_df['medical_specialty'] == specialty]['transcription']
    word_matrix = vectorizer.fit_transform(specialty_texts)
    word_frequencies = word_matrix.sum(axis=0)
    words = vectorizer.get_feature_names_out()
    specialty_word_frequencies[specialty] = dict(zip(words, word_frequencies.tolist()[0]))

# Create a list of all unique words across all specialties
all_unique_words = set(word for freq_dict in specialty_word_frequencies.values() for word in freq_dict.keys())

# Create word frequency array for each specialty
frequencies = []
for specialty, freq_dict in specialty_word_frequencies.items():
    freq_vector = [freq_dict[word] if word in freq_dict else 0 for word in all_unique_words]
    frequencies.append(freq_vector)

# Convert the frequencies list into a numpy array
frequencies_array = np.array(frequencies)

# Calculate cosine similarity between specialties
cosine_similarities = cosine_similarity(frequencies_array)

# Get the specialty names
specialties = list(specialty_word_frequencies.keys())

# Create a heatmap showing the cosine similarity between all specialties
plt.figure(figsize=(13,13))
sns.heatmap(cosine_similarities, annot=True, xticklabels=specialties, yticklabels=specialties, cmap='coolwarm')
plt.title('Cosine Similarity between Medical Specialties')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Mask medical specialties with almost similar symptoms**

In [ ]:
mask = mt_df['medical_specialty'] == ' Neurosurgery'
mt_df.loc[mask, 'medical_specialty'] = ' Neurology'
#mask = mt_df['medical_specialty'] == ' Radiology'
#mt_df.loc[mask, 'medical_specialty'] = ' Neurology'
#mask = mt_df['medical_specialty'] == ' Radiology'
#mt_df.loc[mask, 'medical_specialty'] = ' Cardiovascular / Pulmonary'
mask = mt_df['medical_specialty'] == ' Nephrology'
mt_df.loc[mask, 'medical_specialty'] = ' Urology'

In [ ]:
mtdata_df.head()

In [ ]:
mt_df.info()

In [ ]:
data_categories  = mt_df.groupby(mt_df['medical_specialty'])
i = 1

for catName,dataCategory in data_categories:
    print('Cat:'+str(i)+' '+catName + ' : '+ str(len(dataCategory)) )
    i = i+1

**Remove medical specialty that is less than 50 in number**

In [ ]:
filtdata = data_categories.filter(lambda x:x.shape[0] < 50)
filtdata = filtdata.groupby(filtdata['medical_specialty'])
i=1
for catName,dataCategory in filtdata:
    print('Cat:'+str(i)+' '+catName + ' : '+ str(len(dataCategory)) )
    i = i+1

In [ ]:
mt_df = mt_df[mt_df['medical_specialty'] != ' Dermatology']
mt_df = mt_df[mt_df['medical_specialty'] != ' Podiatry']

In [ ]:
data_categories  = mt_df.groupby(mt_df['medical_specialty'])
i = 1

for catName,dataCategory in data_categories:
    print('Cat:'+str(i)+' '+catName + ' : '+ str(len(dataCategory)) )
    i = i+1

**Tokenization**

In [ ]:
#Tokenization
from nltk.tokenize import word_tokenize

mt_df['tokenized_words'] = mt_df['transcription'].apply(nltk.word_tokenize)
mt_df.head(5)

In [ ]:
mt_df.loc[3, 'transcription']

In [ ]:
mt_df.loc[3, 'tokenized_words']

In [ ]:
mt_df['medical_specialty'].unique()

**POSTag Analysis**

In [ ]:
nltk.download('averaged_perceptron_tagger')

In [ ]:
from nltk import pos_tag

In [ ]:
mt_df['POSTags'] = mt_df['tokenized_words'].apply(pos_tag)

In [ ]:
mt_df.head()

In [ ]:
from collections import Counter

# Flatten the list of POS tuples and extract only the POS tags
all_pos = [pos for pos_list in mt_df['POSTags'] for noun, pos in pos_list]

# Counting POS tag frequencies
pos_counter = Counter(all_pos)

# Extracting POS tag names and counts for plotting
pos_names, pos_counts = zip(*pos_counter.items())

# Creating a bar chart
plt.figure(figsize=(10, 6))  # Adjust the figure size as needed
plt.bar(pos_names, pos_counts)
plt.xlabel('POS Tags')
plt.ylabel('Frequency')
plt.title('POS Tag Frequency Chart')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
postags_list = mt_df['POSTags'].tolist()

# Extract unique POS tags
unique_pos_tags = set(pos_tag for tag_list in postags_list for _, pos_tag in tag_list)

# Print unique POS tags
print("Unique POS tags:", unique_pos_tags)

In [ ]:
mt_df['transcription'][4]

In [ ]:
print(mt_df['POSTags'][4])

In [ ]:
from collections import defaultdict
# Create a dictionary to store words for each POS tag
word_dict = defaultdict(list)

# Populate the dictionary with words for each POS tag
for word, tag in mt_df['POSTags'][4]:
    word_dict[tag].append(word)

# Print the words for each POS tag
for tag, words in word_dict.items():
    print(f"{tag} = {', '.join(words)}")

**Keep Only relevant POS tags**

In [ ]:
# Define the relevant POS tags to keep
relevant_pos_tags = ['NN', 'NNS', 'JJ']

# Function to filter and extract important medical terms
def extract_medical_terms(row):
    terms = []
    for token, pos in row['POSTags']:
        if pos in relevant_pos_tags:
            terms.append(token)
    return terms

# Apply the function to the DataFrame to create a new column 'MedicalTerms'
mt_df['MedicalTerms'] = mt_df.apply(extract_medical_terms, axis=1)

# Display the resulting DataFrame with the extracted medical terms
print(mt_df[['POSTags', 'MedicalTerms']])

In [ ]:
mt_df

In [ ]:
filtered_df = mt_df.copy()

In [ ]:
filtered_df.head()

In [ ]:
# Function to join medical terms into a sentence
def join_medical_terms(row):
    return ' '.join(row['MedicalTerms'])

# Apply the function to the DataFrame to create a single string 'MedicalText'
filtered_df['MedicalText'] = filtered_df.apply(join_medical_terms, axis=1)

In [ ]:
filtered_df.head(5)

In [ ]:
filtered_df.info()

In [ ]:
# List of columns to remove
cols_to_rem = ['POSTags', 'MedicalTerms' , 'tokenized_words']

# Remove the specified columns
filtered_df = filtered_df.drop(columns=cols_to_rem, axis=1)

In [ ]:
filtered_df.head()

In [ ]:
filtered_df.shape

**Vectorization**

In [ ]:
vectorizer = TfidfVectorizer(analyzer='word', stop_words='english', ngram_range=(1, 3), max_df=0.75, min_df=5, use_idf=True, smooth_idf=True, sublinear_tf=True, max_features=1000)
tfIdfMat = vectorizer.fit_transform(filtered_df['MedicalText'].tolist())
feature_names = sorted(vectorizer.get_feature_names_out())

# Print the feature names
print(feature_names)

In [ ]:
print("Shape of TF-IDF matrix:", tfIdfMat.shape)

In [ ]:
print(tfIdfMat.toarray()[:2, :2])  # Print the first 2 rows and 2 columns

In [ ]:
import gc
from sklearn.manifold import TSNE
gc.collect()
tfIdfMatrix = tfIdfMat.todense()
tfIdfArray = np.asarray(tfIdfMatrix)
labels = filtered_df['medical_specialty'].tolist()
tsne_results = TSNE(n_components=2, init='random', random_state=0, perplexity=40).fit_transform(tfIdfArray)
plt.figure(figsize=(20,10))
palette = sns.hls_palette(12, l=.3, s=.9)
sns.scatterplot(
    x=tsne_results[:,0], y=tsne_results[:,1],
    hue=labels,
    palette= palette,
    legend="full",
    alpha=0.3
)
plt.show()

**Dimensionality reduction using PCA**

In [ ]:
pca = PCA(n_components=0.95)
tfIdfMat_reduced = pca.fit_transform(tfIdfMat.toarray())
labels = filtered_df['medical_specialty'].tolist()
category_list = filtered_df.medical_specialty.unique()
category_list

In [ ]:
tfIdfMat_reduced.shape

In [ ]:
import gc
from sklearn.manifold import TSNE
# Perform t-SNE
tfIdfArray = np.asarray(tfIdfMat_reduced)  # Use the reduced PCA matrix
labels = filtered_df['medical_specialty'].tolist()
tsne_results = TSNE(n_components=2, init='random', random_state=0, perplexity=40).fit_transform(tfIdfArray)

# Plot the t-SNE results
plt.figure(figsize=(20, 10))
palette = sns.color_palette("tab20", n_colors=len(category_list))
sns.scatterplot(
    x=tsne_results[:, 0], y=tsne_results[:, 1],
    hue=labels,
    palette=palette,
    legend="full",
    alpha=0.3
)
plt.title('t-SNE Visualization of Medical Specialties')
plt.show()

**Reduce class imbalance using SMOTE**

In [ ]:
smote_over_sample = SMOTE(sampling_strategy='minority')
labels = filtered_df['medical_specialty'].tolist()
X, y = smote_over_sample.fit_resample(tfIdfMat_reduced, labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y,random_state=1)
print('Train_Set_Size:'+str(X_train.shape))
print('Test_Set_Size:'+str(X_test.shape))

In [ ]:
X.shape

In [ ]:
original_class_counts = pd.Series(labels).value_counts()
print("Original class distribution:")
print(original_class_counts)

In [ ]:
smote_class_counts_train = pd.Series(y_train).value_counts()
print("SMOTE-generated class distribution in training set:")
print(smote_class_counts_train)

smote_class_counts_test = pd.Series(y_test).value_counts()
print("SMOTE-generated class distribution in test set:")
print(smote_class_counts_test)

In [ ]:
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
original_class_counts.plot(kind='bar')
plt.title("Original Class Distribution")
plt.xlabel("Medical Specialty")
plt.ylabel("Count")

plt.subplot(1, 2, 2)
smote_class_counts_train.plot(kind='bar', label='Train')
smote_class_counts_test.plot(kind='bar', alpha=0.7, label='Test')
plt.title("SMOTE-generated Class Distribution")
plt.xlabel("Medical Specialty")
plt.ylabel("Count")
plt.legend()

plt.tight_layout()
plt.show()

**Application of Machine Learning to check and compare the accuracy**

**Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Encode department labels into numerical values
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(filtered_df['medical_specialty'])


# Train the Random Forest classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Get user input and preprocess
user_input = input("Enter a medical transcription: ")
process_user_input = process_Text(user_input)

# Transform the user input using the trained TF-IDF vectorizer and PCA
user_input_tfidf = vectorizer.transform([process_user_input])
user_input_pca = pca.transform(user_input_tfidf.toarray())

# Predict the medical department for the user input
predicted_department = clf.predict(user_input_pca)


print("Predicted Medical Department:", predicted_department[0])

In [ ]:
# Evaluate the accuracy of the prediction
accuracy = clf.score(X_test, y_test)
print("Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import classification_report
y_pred = clf.predict(X_test)
# Print the classification report
class_names = label_encoder.classes_
classification_rep = classification_report(y_test, y_pred, target_names=class_names)
print("Classification Report For Random Forest:\n", classification_rep)

**Logistic regression**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Encode department labels into numerical values
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(filtered_df['medical_specialty'])

# Train the Logistic Regression model
clf2 = LogisticRegression(max_iter=1000, random_state=42)
clf2.fit(X_train, y_train)

# Get user input and preprocess
user_input = input("Enter a medical transcription: ")
#clean_user_input = clean_text(user_input)
#lemmatize_user_input = lemmatize_text(clean_user_input)
process_user_input = process_Text(user_input)

# Transform the user input using the trained TF-IDF vectorizer
user_input_tfidf = vectorizer.transform([process_user_input])
user_input_pca = pca.transform(user_input_tfidf.toarray())

# Predict the medical department for the user input
predicted_department = clf2.predict(user_input_pca)

print("Predicted Medical Department:", predicted_department[0])

In [ ]:
# Evaluate the accuracy of the prediction
accuracy = clf2.score(X_test, y_test)
print("Accuracy:", accuracy)

In [ ]:
# Get predictions for the test set
y_pred2 = clf2.predict(X_test)
# Print the classification report for Logistic Regression
classification_rep2 = classification_report(y_test, y_pred2, target_names=class_names)
print("Classification Report for Logistic Regression:\n", classification_rep2)